# Q1: Funnel

Where do visitors drop off between viewing a product and purchasing?
Uses the cumulative funnel (q1d in sql/03_funnel.sql): each step counts sessions that reached at least that step.

In [1]:
import warnings
warnings.filterwarnings("ignore", message="BigQuery Storage module not found")

from pathlib import Path
from google.cloud import bigquery

client = bigquery.Client(project="ga4-funnel-analysis-511719")
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

In [2]:
sql = """
select
    count(*) as sessions,
    countif(viewed_item or added_to_cart or began_checkout or purchased) as reached_view,
    countif(added_to_cart or began_checkout or purchased) as reached_cart,
    countif(began_checkout or purchased) as reached_checkout,
    countif(purchased) as reached_purchase
from `ga4-funnel-analysis-511719.ga4_analysis.session_base`
"""

wide = client.query(sql).to_dataframe()
wide

,sessions,reached_view,reached_cart,reached_checkout,reached_purchase
0,360129,77314,20333,11106,4044


In [3]:
funnel = wide.melt(var_name="step", value_name="session_count")
funnel["pct_of_previous"] = (funnel["session_count"] / funnel["session_count"].shift(1)).round(4)
funnel["pct_of_all"] = (funnel["session_count"] / funnel["session_count"].iloc[0]).round(4)
funnel

,step,session_count,pct_of_previous,pct_of_all
0,sessions,360129,<NA>,1.0
1,reached_view,77314,0.2147,0.2147
2,reached_cart,20333,0.263,0.0565
3,reached_checkout,11106,0.5462,0.0308
4,reached_purchase,4044,0.3641,0.0112


In [4]:
funnel.to_csv(output_dir / "q1_funnel.csv", index=False)